#### This is for creating a dashboard that shows the past 40 years of NBA player's performance (overall season stats in all areas) for each MVP and their runner up. Compare those stats over the past 40 years.

In [29]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def get_mvp_data(start_year=1985, end_year=2025):
    """
    Scrapes Basketball Reference for MVP and runner-up for each season.
    start_year = the year the season ended (e.g. 1985 = 1984-85 season)
    """
    mvp_data = []

    for year in range(start_year, end_year + 1):
        url = f"https://www.basketball-reference.com/awards/awards_{year}.html"
        
        try:
            headers = {"User-Agent": "Mozilla/5.0"}
            response = requests.get(url, headers=headers, timeout=10)
            response.encoding = "utf-8"
            soup = BeautifulSoup(response.text, "html.parser")

            # Find the MVP voting table
            table = soup.find("table", {"id": "mvp"})
            if not table:
                print(f"  No MVP table found for {year}")
                continue

            rows = table.find("tbody").find_all("tr")
            players = []
            for row in rows:
                name_cell = row.find("td", {"data-stat": "player"})
                if name_cell and name_cell.text.strip():
                    players.append(name_cell.text.strip())
                if len(players) == 2:
                    break

            if len(players) >= 2:
                season = f"{year-1}-{str(year)[2:]}"
                mvp_data.append({
                    "season": season,
                    "mvp": players[0],
                    "runner_up": players[1]
                })
                print(f"  {season}: MVP={players[0]}, Runner-Up={players[1]}")
            
            time.sleep(3)  # Be respectful to Basketball Reference servers

        except Exception as e:
            print(f"  Error fetching {year}: {e}")

    return mvp_data

print("Scraping MVP data from Basketball Reference...")
MVP_DATA = get_mvp_data(start_year=1985, end_year=2025)
print(f"\nLoaded {len(MVP_DATA)} seasons")

Scraping MVP data from Basketball Reference...
  1984-85: MVP=Larry Bird, Runner-Up=Magic Johnson
  1985-86: MVP=Larry Bird, Runner-Up=Dominique Wilkins
  1986-87: MVP=Magic Johnson, Runner-Up=Michael Jordan
  1987-88: MVP=Michael Jordan, Runner-Up=Larry Bird
  1988-89: MVP=Magic Johnson, Runner-Up=Michael Jordan
  1989-90: MVP=Magic Johnson, Runner-Up=Charles Barkley
  1990-91: MVP=Michael Jordan, Runner-Up=Magic Johnson
  1991-92: MVP=Michael Jordan, Runner-Up=Clyde Drexler
  1992-93: MVP=Charles Barkley, Runner-Up=Hakeem Olajuwon
  1993-94: MVP=Hakeem Olajuwon, Runner-Up=David Robinson
  1994-95: MVP=David Robinson, Runner-Up=Shaquille O'Neal
  1995-96: MVP=Michael Jordan, Runner-Up=David Robinson
  1996-97: MVP=Karl Malone, Runner-Up=Michael Jordan
  1997-98: MVP=Michael Jordan, Runner-Up=Karl Malone
  1998-99: MVP=Karl Malone, Runner-Up=Alonzo Mourning
  1999-00: MVP=Shaquille O'Neal, Runner-Up=Kevin Garnett
  2000-01: MVP=Allen Iverson, Runner-Up=Tim Duncan
  2001-02: MVP=Tim Dun

In [2]:
import pandas as pd
MVP_DATA = pd.read_csv('MVP_Data.csv').to_dict(orient='records')
print(f"Loaded {len(MVP_DATA)} seasons")

Loaded 41 seasons


In [3]:
from nba_api.stats.static import players
from nba_api.stats.endpoints import playercareerstats
import pandas as pd
import time
import sys

def get_player_id(name):
    # Normalize special characters for matching
    import unicodedata
    def normalize(s):
        return unicodedata.normalize('NFD', s).encode('ascii', 'ignore').decode('utf-8').lower()
    
    all_players = players.get_players()
    for player in all_players:
        if normalize(player['full_name']) == normalize(name):
            return player['id']
    return None

def get_season_stats(player_name, season):
    player_id = get_player_id(player_name)
    if not player_id:
        print(f"  Could not find player: {player_name}")
        return None

    time.sleep(0.6)

    career = playercareerstats.PlayerCareerStats(player_id=player_id)
    df = career.season_totals_regular_season.get_data_frame()

    row = df[df['SEASON_ID'] == season]
    if row.empty:
        print(f"  No stats found for {player_name} in {season}")
        return None

    r = row.iloc[0]
    gp = r['GP'] if r['GP'] > 0 else 1

    return {
        "PLAYER":   player_name,
        "TEAM":     r['TEAM_ABBREVIATION'],
        "SEASON":   season,
        "PTS":      round(r['PTS'] / gp, 1),
        "AST":      round(r['AST'] / gp, 1),
        "REB":      round(r['REB'] / gp, 1),
        "STL":      round(r['STL'] / gp, 1),
        "BLK":      round(r['BLK'] / gp, 1),
        "FG_PCT":   round(r['FG_PCT'] * 100, 1),
        "FG3_PCT":  round(r['FG3_PCT'] * 100, 1) if r['FG3_PCT'] else 0,
        "FT_PCT":   round(r['FT_PCT'] * 100, 1),
        "TOV":      round(r['TOV'] / gp, 1),
        "GP":       int(r['GP']),
        "MIN":      round(r['MIN'] / gp, 1),
    }

print("Functions ready!")

Functions ready!


In [4]:
rows = []

for entry in MVP_DATA:
    season = entry['season']
    print(f'Fetching {season}...')

    for role, name in [("MVP", entry["mvp"]), ("Runner-Up", entry["runner_up"])]:
        stats = get_season_stats(name, season)
        if stats:
            stats['ROLE'] = role
            rows.append(stats)
    
df = pd.DataFrame(rows)
df.to_csv('C:/Users/Owner/OneDrive/Desktop/Code/mvp_stats.csv', index=False)

print(f"/nDone! Saved {len(rows)} rows to mvp_stats.csv")

Fetching 1984-85...
Fetching 1985-86...
Fetching 1986-87...
Fetching 1987-88...
Fetching 1988-89...
Fetching 1989-90...
Fetching 1990-91...
Fetching 1991-92...
Fetching 1992-93...
Fetching 1993-94...
Fetching 1994-95...
Fetching 1995-96...
Fetching 1996-97...
Fetching 1997-98...
Fetching 1998-99...
Fetching 1999-00...
Fetching 2000-01...
Fetching 2001-02...
Fetching 2002-03...
Fetching 2003-04...
Fetching 2004-05...
Fetching 2005-06...
Fetching 2006-07...
Fetching 2007-08...
Fetching 2008-09...
Fetching 2009-10...
Fetching 2010-11...
Fetching 2011-12...
Fetching 2012-13...
Fetching 2013-14...
Fetching 2014-15...
Fetching 2015-16...
Fetching 2016-17...
Fetching 2017-18...
Fetching 2018-19...
Fetching 2019-20...
Fetching 2020-21...
Fetching 2021-22...
Fetching 2022-23...
Fetching 2023-24...
Fetching 2024-25...
/nDone! Saved 82 rows to mvp_stats.csv


In [5]:
# previewing the data
df = pd.read_csv("C:/Users/Owner/OneDrive/Desktop/Code/mvp_stats.csv")
print(df.columns.tolist())
print(df.head())

['PLAYER', 'TEAM', 'SEASON', 'PTS', 'AST', 'REB', 'STL', 'BLK', 'FG_PCT', 'FG3_PCT', 'FT_PCT', 'TOV', 'GP', 'MIN', 'ROLE']
              PLAYER TEAM   SEASON   PTS   AST   REB  STL  BLK  FG_PCT  \
0         Larry Bird  BOS  1984-85  28.7   6.6  10.5  1.6  1.2    52.2   
1      Magic Johnson  LAL  1984-85  18.3  12.6   6.2  1.5  0.3    56.1   
2         Larry Bird  BOS  1985-86  25.8   6.8   9.8  2.0  0.6    49.6   
3  Dominique Wilkins  ATL  1985-86  30.3   2.6   7.9  1.8  0.6    46.8   
4      Magic Johnson  LAL  1986-87  23.9  12.2   6.3  1.7  0.4    52.2   

   FG3_PCT  FT_PCT  TOV  GP   MIN       ROLE  
0     42.7    88.2  3.1  80  39.5        MVP  
1     18.9    84.3  4.0  77  36.1  Runner-Up  
2     42.3    89.6  3.2  82  38.0        MVP  
3     18.6    81.8  3.2  78  39.1  Runner-Up  
4     20.5    84.8  3.8  80  36.3        MVP  


In [6]:
# Idea: For each MVP season, rank how well the team did when that MVP was off the court vs. on the court. Rank the top 10 players throughout all of these seasons. 
from nba_api.stats.endpoints import playergamelog
from nba_api.stats.static import players as nba_players
import pandas as pd
import time

def get_on_off_wins(player_name, season):
    """
    Approximates on/off impact using team win % in games played vs missed.
    season format: '2023-24'
    """
    player_id = get_player_id(player_name)
    if not player_id:
        print(f"Could not find player: {player_name}")
        return None

    time.sleep(0.6)

    # Get player game log for the season
    gamelog = playergamelog.PlayerGameLog(
        player_id=player_id,
        season=season
    )
    df = gamelog.get_data_frames()[0]

    if df.empty:
        print(f"No game log found for {player_name} in {season}")
        return None

    # Games the player played
    games_played = len(df)
    wins_with = len(df[df['WL'] == 'W'])
    win_pct_with = round((wins_with / games_played) * 100, 1)

    # Get team game log to find games player missed
    from nba_api.stats.endpoints import teamgamelog
    from nba_api.stats.static import teams as nba_teams

    # team_abbr = df['MATCHUP'].iloc[0].split(' ')[0]
    # all_teams = nba_teams.get_teams()
    # team = next((t for t in all_teams if t['abbreviation'] == team_abbr), None)

    team_abbr = df['MATCHUP'].iloc[0].split(' ')[0]

    # Map old/alternate abbreviations to current nba_api ones
    abbr_map = {
        'SAN': 'SAS',  # San Antonio Spurs
        'PHO': 'PHX',  # Phoenix Suns
        'NJN': 'BKN',  # New Jersey Nets -> Brooklyn Nets
        'SEA': 'OKC',  # Seattle SuperSonics -> OKC Thunder
        'NOH': 'NOP',  # New Orleans Hornets -> Pelicans
        'NOK': 'NOP',  # New Orleans/Oklahoma City
        'VAN': 'MEM',  # Vancouver Grizzlies -> Memphis
        'NJN': 'BKN',  # New Jersey Nets
        'WSB': 'WAS',  # Washington Bullets -> Wizards
    }

    team_abbr = abbr_map.get(team_abbr, team_abbr)
    all_teams = nba_teams.get_teams()
    team = next((t for t in all_teams if t['abbreviation'] == team_abbr), None)

    if not team:
        print(f"Could not find team: {team_abbr}")
        return None

    time.sleep(0.6)

    team_log = teamgamelog.TeamGameLog(
        team_id=team['id'],
        season=season
    )
    team_df = team_log.get_data_frames()[0]

    # Games player missed = team games not in player game log
    player_game_ids = set(df['Game_ID'])
    missed_df = team_df[~team_df['Game_ID'].isin(player_game_ids)]

    games_missed = len(missed_df)
    wins_without = len(missed_df[missed_df['WL'] == 'W'])
    win_pct_without = round((wins_without / games_missed) * 100, 1) if games_missed > 0 else None

    return {
        "player": player_name,
        "season": season,
        "games_played": games_played,
        "win_pct_with": win_pct_with,
        "games_missed": games_missed,
        "win_pct_without": win_pct_without,
        "win_pct_diff": round(win_pct_with - win_pct_without, 1) if win_pct_without is not None else None
    }

# Test with one player first
result = get_on_off_wins("LeBron James", "2012-13")
print(result)

{'player': 'LeBron James', 'season': '2012-13', 'games_played': 76, 'win_pct_with': 80.3, 'games_missed': 6, 'win_pct_without': 83.3, 'win_pct_diff': -3.0}


In [7]:
onoff_rows = []

for entry in MVP_DATA:
    season = entry['season']
    player = entry['mvp']
    print(f"Fetching on/off for {player} ({season})...")
    
    result = get_on_off_wins(player, season)
    if result:
        onoff_rows.append(result)

onoff_df = pd.DataFrame(onoff_rows)
onoff_df.to_csv('C:/Users/Owner/OneDrive/Desktop/Code/mvp_onoff.csv', index=False)
print(f"\nDone! Saved {len(onoff_rows)} rows to mvp_onoff.csv")

Fetching on/off for Larry Bird (1984-85)...
Fetching on/off for Larry Bird (1985-86)...
Fetching on/off for Magic Johnson (1986-87)...
Fetching on/off for Michael Jordan (1987-88)...
Fetching on/off for Magic Johnson (1988-89)...
Fetching on/off for Magic Johnson (1989-90)...
Fetching on/off for Michael Jordan (1990-91)...
Fetching on/off for Michael Jordan (1991-92)...
Fetching on/off for Charles Barkley (1992-93)...
Fetching on/off for Hakeem Olajuwon (1993-94)...
Fetching on/off for David Robinson (1994-95)...
Fetching on/off for Michael Jordan (1995-96)...
Fetching on/off for Karl Malone (1996-97)...
Fetching on/off for Michael Jordan (1997-98)...
Fetching on/off for Karl Malone (1998-99)...
Fetching on/off for Shaquille O'Neal (1999-00)...
Fetching on/off for Allen Iverson (2000-01)...
Fetching on/off for Tim Duncan (2001-02)...
Fetching on/off for Tim Duncan (2002-03)...
Fetching on/off for Kevin Garnett (2003-04)...
Fetching on/off for Steve Nash (2004-05)...
Fetching on/off for

In [ ]:
# Creating a database table for this.
import sqlite3

# Create database
conn = sqlite3.connect('C:/Users/Owner/OneDrive/Desktop/Code/nba_mvp.db')
cursor = conn.cursor()

# Create tables
cursor.executescript('''
    CREATE TABLE IF NOT EXISTS mvp_list (
        season TEXT PRIMARY KEY,
        mvp TEXT,
        runner_up TEXT
    );

    CREATE TABLE IF NOT EXISTS player_stats (
        player_id INTEGER PRIMARY KEY AUTOINCREMENT,
        player TEXT,
        team TEXT,
        season TEXT,
        role TEXT,
        pts REAL,
        ast REAL,
        reb REAL,
        stl REAL,
        blk REAL,
        fg_pct REAL,
        fg3_pct REAL,
        ft_pct REAL,
        tov REAL,
        gp INTEGER,
        min REAL
    );

    CREATE TABLE IF NOT EXISTS onoff_stats (
        player_id INTEGER PRIMARY KEY AUTOINCREMENT,
        player TEXT,
        season TEXT,
        games_played INTEGER,
        win_pct_with REAL,
        games_missed INTEGER,
        win_pct_without REAL,
        win_pct_diff REAL
    );
''')

conn.commit()
conn.close()
print("Database and tables created!")

Database and tables created!


In [11]:
conn = sqlite3.connect('C:/Users/Owner/OneDrive/Desktop/Code/nba_mvp.db')

# Load CSVs
mvp_list_df = pd.DataFrame(MVP_DATA)
player_stats_df = pd.read_csv('C:/Users/Owner/OneDrive/Desktop/Code/mvp_stats.csv')
onoff_df = pd.read_csv('C:/Users/Owner/OneDrive/Desktop/Code/mvp_onoff.csv')

# Insert into database — if_exists='replace' clears and repopulates each time
mvp_list_df.to_sql('mvp_list', conn, if_exists='replace', index=False)
player_stats_df.to_sql('player_stats', conn, if_exists='replace', index=False)
onoff_df.to_sql('onoff_stats', conn, if_exists='replace', index=False)

conn.commit()
conn.close()
print("Data inserted into database!")

Data inserted into database!


In [ ]:
# Run this command in powershell to launch streamlit: 
# cd C:\Users\Owner\OneDrive\Desktop\Code
# streamlit run app.py